In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer — Metadata-Driven Aggregation
# MAGIC Reads a Silver table and produces a business-level aggregate in Gold.
# MAGIC The grouping columns, aggregation expressions, and source/target tables
# MAGIC are all parameterized so one notebook serves many Gold aggregates.

# COMMAND ----------

dbutils.widgets.text("source_catalog", "main")
dbutils.widgets.text("source_schema", "silver")
dbutils.widgets.text("source_table", "orders")
dbutils.widgets.text("target_catalog", "main")
dbutils.widgets.text("target_schema", "gold")
dbutils.widgets.text("target_table", "daily_sales_by_region")
dbutils.widgets.text("group_by_columns", "order_date,region")             # comma-separated
dbutils.widgets.text("agg_expressions", "sum(order_amount) as total_sales,count(order_id) as order_count")

source_catalog    = dbutils.widgets.get("source_catalog")
source_schema     = dbutils.widgets.get("source_schema")
source_table      = dbutils.widgets.get("source_table")
target_catalog    = dbutils.widgets.get("target_catalog")
target_schema     = dbutils.widgets.get("target_schema")
target_table      = dbutils.widgets.get("target_table")
group_by_columns  = [c.strip() for c in dbutils.widgets.get("group_by_columns").split(",")]
agg_expressions   = [e.strip() for e in dbutils.widgets.get("agg_expressions").split(",")]

source_fqn = f"{source_catalog}.{source_schema}.{source_table}"
target_fqn = f"{target_catalog}.{target_schema}.{target_table}"

print(f"Source table   : {source_fqn}")
print(f"Target table   : {target_fqn}")
print(f"Group by       : {group_by_columns}")
print(f"Aggregations   : {agg_expressions}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Optional metadata lookup
# MAGIC Grouping columns and aggregation logic could equally live in
# MAGIC `main.control.pipeline_metadata`, keyed by `target_table`, so adding a
# MAGIC new Gold mart is a metadata insert, not a code change.

# COMMAND ----------

metadata_df = (
    spark.table("main.control.pipeline_metadata")
    .filter(f"layer = 'gold' AND target_table = '{target_table}'")
)

if metadata_df.count() > 0:
    row = metadata_df.collect()[0].asDict()
    source_fqn = row.get("source_fqn", source_fqn)
    target_fqn = row.get("target_fqn", target_fqn)
    group_by_columns = row.get("group_by_columns", ",".join(group_by_columns)).split(",")
    agg_expressions = row.get("agg_expressions", ",".join(agg_expressions)).split(",")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Aggregate Silver -> Gold

# COMMAND ----------

silver_df = spark.table(source_fqn)

silver_df.createOrReplaceTempView("silver_src")

agg_sql = f"""
SELECT
    {", ".join(group_by_columns)},
    {", ".join(agg_expressions)}
FROM silver_src
GROUP BY {", ".join(group_by_columns)}
"""

gold_df = spark.sql(agg_sql)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write Gold table (full overwrite — typical for aggregates)

# COMMAND ----------

(
    gold_df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(target_fqn)
)

print(f"Gold aggregate complete: {target_fqn}")
